# Cual es el problema?

Abandono de clientes (Churn) en una empresa de telecomunicaciones. El objetivo es predecir si un cliente abandonará el servicio o no, utilizando un modelo de regresión logística.

# Por que es importante?

Predecir el abandono de clientes es crucial para las empresas, ya que retener a los clientes existentes suele ser más rentable que adquirir nuevos. Al identificar a los clientes que tienen una alta probabilidad de abandonar, la empresa puede implementar estrategias de retención personalizadas, como ofertas especiales o mejoras en el servicio, para mantener a esos clientes satisfechos y reducir la tasa de abandono.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt

# Cargar el dataset
data = pd.read_csv('abandono_clientes.csv')

print(f"Filas: {data.shape[0]}, Columnas: {data.shape[1]}")

print(f"Tipo de datos:\n{data.dtypes}")

print(f"Valores nulos:\n{data.isnull().sum()}")




In [ ]:
data.describe().round(2)

In [ ]:
# Distribución de la variable objetivo
# Si el 99% de los clientes no abandonan, es un modelo que tendra una precision (accuracy) completamente inutil
# porque siempre va a predecir que el cliente no abandona, y acertara el 99% de las veces, pero no estara prediciendo nada realmente
plt.figure(figsize=(6, 4))
data['abandono'].value_counts().plot(kind='bar')
# Porcentaje de clientes que abandonan
abandono_percentage = data['abandono'].value_counts(normalize=True) * 100

for i, v in enumerate(abandono_percentage):
    plt.text(i, v + 0.5, f'{v:.1f}% \n {int(data["abandono"].value_counts()[i])}', ha='center', fontweight='bold')

plt.title('Distribución de Abandono de Clientes (%)')
plt.xlabel('Abandono')
plt.ylabel('Cantidad de Clientes')
plt.xticks(rotation=0)
plt.show()

print("Distribucion de la variable objetivo (abandono):")
print(f"  Permanece (0): {data['abandono'].value_counts()[0]} clientes ({abandono_percentage[0]:.1f}%)")
print(f"  Abandona  (1): {data['abandono'].value_counts()[1]} clientes ({abandono_percentage[1]:.1f}%)")
print()
print("Las clases estan moderadamente desbalanceadas.")
print("Hay mas clientes que permanecen que clientes que abandonan.")
print("Esto es lo tipico en la realidad: la mayoria de clientes NO se van.")


# Analisis exploratorio de datos (EDA): como se comportan las variables segun abandono o no abandono?

#### Antes de entrenar el modelo, veamos si las variables tienen relación con el abandono. Esto nos ayuda a entender que factores podrían influir en la decisión del cliente.

In [ ]:
# Comparar el promedio de cada variable numerica entre los clientes que abandonan y los que permanecen

comparacion = data.groupby('abandono')[['antiguedad_meses', 
                                       'factura_mensual', 
                                       'llamadas_soporte', 
                                       'satisfaccion']].mean().round(2)
comparacion.index = ['Permanece (0)', 'Abandona (1)']
print("Comparacion de promedios entre clientes que abandonan (1) y los que permanecen (0):")
comparacion

In [ ]:
# Analisis del tipo de contrato vs abandono
contrato_abandono = pd.crosstab(data['contrato'], data['abandono'], normalize='index') * 100
contrato_abandono.columns = ['Permanece (0)', 'Abandona (1)']
print("\nPorcentaje de abandono por tipo de contrato:")
display(contrato_abandono.round(2))
contrato_abandono.plot(kind='bar', stacked=True, figsize=(8, 5))
plt.title('Porcentaje de Abandono por Tipo de Contrato')
plt.xlabel('Tipo de Contrato')
plt.ylabel('Porcentaje de Clientes')
plt.legend(title='Abandono', labels=['Permanece (0)', 'Abandona (1)'])
plt.xticks(rotation=0)
plt.show()


# Preparar los datos para el modelo

1. Separar variables predictoras (X) de la variable objetivo (y).
2. Entrenar los datos con (80%) del dataset y probar con el (20%) restante.



In [ ]:
X = data[['antiguedad_meses', 
          'factura_mensual', 
          'llamadas_soporte', 
          'satisfaccion', 
          'contrato']]
y = data['abandono']

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42,
                                                    stratify=y
                                                    )
print(f"Datos de entrenamiento: {X_train.shape[0]} filas, {X_train.shape[1]} columnas")
print(f"Datos de prueba: {X_test.shape[0]} filas, {X_test.shape[1]} columnas")
print(f"Proporcion de abandono en entrenamiento: {y_train.mean():.2f}")
print(f"Proporcion de abandono en prueba: {y_test.mean():.2f}")

In [ ]:
numeric_features = ['antiguedad_meses', 'factura_mensual', 'llamadas_soporte', 'satisfaccion']
categorical_features = ['contrato']
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first'))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # ('classifier', LogisticRegression(random_state=42))
    ('model', LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)
print("Modelo entrenado exitosamente.")


# Hacer Predicciones y evaluar el modelo

- Usamos el modelo entrenado para predecir si los clientes en el conjunto de prueba abandonarán o no.
- Evaluamos el rendimiento del modelo utilizando métricas como la precisión, la matriz de confusión y el informe de clasificación, que nos indican qué tan bien el modelo está clasificando a los clientes en las categorías de abandono y no abandono.

In [ ]:
# Generar predicciones y evaluar el modelo
y_pred = model.predict(X_test)

# Mostrar las primeras 10 predicciones junto con los valores reales
print("Predicciones vs Valores Reales")
comparacion_pred = pd.DataFrame({'Valor Real': y_test.values, 'Predicción': y_pred, 'Correcto': y_test.values == y_pred})
display(comparacion_pred.head(10))

# Evaluacion del modelo: Exactitud (Acurracy), Matriz de confusión y Reporte de clasificación
Cuando hablamos del porcentaje de presición nos referimos al % de predicciones que fueron correctas.

Accuraracy = Predicciones correctas / Total de predicciones

Ejemplo: Si el 95% de cliente **No** abandonan, un modelo que siempre predice "No abandono" tendrá una precisión del 95%, pero no será útil para identificar a los clientes que realmente abandonarán. Por eso, es importante usar otras métricas como la matriz de confusión y el informe de clasificación para evaluar el rendimiento del modelo de manera más completa.


In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.2f} ({acc*100:.1f}%)')
print(f"Esto significa que el modelo esta acertando el {acc*100:.1f}% de las veces,\n lo cual es bastante bueno considerando que la clase mayoritaria (no abandono) representa el 70% de los casos.")
print(f"="*100)
print(f"Esto tambien indica que el modelo acerto {int(acc*len(y_test))} de {len(y_test)} predicciones correctamente.")


In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')
print(f'Confusion Matrix:\n{conf_matrix}')
print(f'Classification Report:\n{class_report}')

In [ ]:
# Preprocesamiento de datos
# Seleccionar características y variable objetivo
# columna 'contrato' es categórica, por lo que se aplicará One-Hot Encoding
pipe = ColumnTransformer(transformers=[
    ('encoder', OneHotEncoder(), ['contrato'])
], remainder='passthrough')

pipe = Pipeline(steps=[('preprocessor', pipe)])

X = pipe.fit_transform(data.drop(['abandono'], axis=1))
y = data['abandono']
# Dividir el dataset en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Crear y entrenar el modelo de regresión logística
model = LogisticRegression()
model.fit(X_train, y_train)
# Realizar predicciones
y_pred = model.predict(X_test)
# Evaluar el modelo
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'Mean Squared Error: {mse:.2f}')
print(f'R^2 Score: {r2:.2f}')

# Evaluar el modelo de clasificación
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')
print(f'Confusion Matrix:\n{conf_matrix}')
print(f'Classification Report:\n{class_report}')


In [ ]:
# Visualización de resultados
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel('Valores Reales')
plt.ylabel('Valores Predichos')
plt.title('Valores Reales vs Predichos')
plt.show()